# Regresi robust (Huber/RANSAC/Theil-Sen)
**tabular › regression_advanced** · notebook `robust_regressors`

**Kapan dipakai:** Target/fitur mengandung outlier ekstrem yang merusak regresi linear biasa.

**Jebakan umum:** Notebook ini menyuntik outlier sintetis pada training agar terlihat efeknya (ditandai jelas). Theil-Sen lambat untuk fitur banyak.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular/regression/diamonds_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular/regression/diamonds_test.csv"      # None kalau tidak ada file test
TARGET     = "price"
ID_COL     = 'id'
DROP_COLS  = []
GROUP_COL  = None                      # isi bila ada grup agar tidak bocor
SCORING    = "neg_root_mean_squared_error"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "robust_regressors"
TASK       = "regression"
KIND       = "tree"
OUT = out_dir("tabular", "regression_advanced", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
y = df[TARGET].to_numpy(dtype=float)
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Model dasar

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.base import clone
lgbm = LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbose=-1)
def fit_eval(name, model_, kind="tree", ytr_=None, store=None):
    pp = build_pipeline(model_, NUM, CAT, kind=kind).fit(Xtr, ytr if ytr_ is None else ytr_)
    pr = pp.predict(Xva)
    if store is not None: store[name] = pr
    return {"model": name, **regression_metrics(yva, pr)}

## 5. Suntik outlier pada training (SINTETIK, hanya untuk demonstrasi)
5% target training dikalikan 10. Validation tetap bersih. Lihat model mana yang tahan.

In [ ]:
from sklearn.linear_model import HuberRegressor, RANSACRegressor, TheilSenRegressor
rng = np.random.default_rng(SEED); ytr_c = ytr.copy()
idx = rng.choice(len(ytr), size=int(0.05 * len(ytr)), replace=False); ytr_c[idx] *= 10
MODELS = {"ridge": Ridge(alpha=1.0), "huber": HuberRegressor(epsilon=1.35, alpha=1e-3, max_iter=1000),
          "ransac": RANSACRegressor(random_state=SEED), "theil-sen": TheilSenRegressor(random_state=SEED, max_subpopulation=3000, n_jobs=-1)}
rows = []
for name, mm in MODELS.items():
    rows.append({**fit_eval(name, clone(mm), kind="linear"), "data_latih": "bersih"})
    rows.append({**fit_eval(name, clone(mm), kind="linear", ytr_=ytr_c), "data_latih": "5% outlier"})
res = pd.DataFrame(rows)
display(res.pivot(index="model", columns="data_latih", values="rmse").assign(naik=lambda d: d["5% outlier"] / d["bersih"]).round(2))

Rasio `naik` mendekati 1 = tahan outlier. Ridge biasanya memburuk paling parah. Pada data nyata, **periksa dulu apakah outlier itu galat atau sinyal** sebelum memakai model robust.

In [ ]:
try:
    save_json({"method": MODEL_KEY, "valid": res.reset_index().to_dict("records") if "res" in dir() else metrics}, OUT / "metrics.json")
except NameError:
    save_json({"method": MODEL_KEY, "valid": metrics}, OUT / "metrics.json")
print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan metrik pada skala yang dipakai juri lomba, bukan hanya RMSE.
- Hasil terbaik biasanya digabung ke ensemble (lihat `notebooks/tabular/ensembling_regression/`).